# 05 · Decision model vs LLM-only routing: latency and cost

Notebook 04 put a trained decision model in the router node of a LangGraph workflow. The common alternative is to let
an **LLM make that decision** too. This notebook compares the two on the same graph and the same tickets:

| router | how it decides |
|---|---|
| **Decision model** | XGBoost on CPU (and optionally our fine-tuned Laya on GPU) |
| **Gemini only** | Gemini with structured JSON output, a written routing policy and 12 labelled examples |
| **Hybrid** | the decision model answers; only tickets where it's unsure go to Gemini |

For each router we measure:
- **quality**: correct destination (the node the ticket lands in) and exact match (all three fields right)
- **latency**: router-node p50 / p95 and whole-graph p50 / p95
- **cost**: tokens per call and **$ per 1,000 tickets**, plus how often the LLM's answer was unusable

The specialist agents stay stubs, so the only thing that changes between runs is the routing decision.

**Runtime:** CPU is enough for XGBoost + Gemini. Turn on `USE_LAYA` (and a T4 GPU) to include Laya.
**Needs:** a Gemini API key (from https://aistudio.google.com/apikey).

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

Install the Gemini SDK (`google-genai`) and LangGraph. Add `laya` if you'll use it.

In [ ]:
%pip install -q google-genai langgraph scikit-learn xgboost pandas matplotlib

## 1. API key

On Colab, add the key once under the **🔑 Secrets** tab (left sidebar) with the name `GEMINI_API_KEY` and allow
notebook access. Otherwise you'll be prompted for it. The key is never printed or saved in the notebook.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    if not os.environ.get("GEMINI_API_KEY"):
        from getpass import getpass
        os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")
print("key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

## 2. Settings

- `GEMINI_MODEL`: `gemini-3.1-flash-lite` is the cheapest current 3-series model ($0.25 in / $1.50 out per 1M tokens).
  Try `gemini-3.8-flash` for a stronger (pricier) LLM. Prices live in `GEMINI_PRICES_PER_1M` in `src/models/llm_router.py`
  (checked 2026-10-19 on the official pricing page; check again before quoting numbers).
- `N_TICKETS`: how many test tickets go through every router. 100 keeps the run around 10–15 minutes on a free key.
- `FREE_TIER`: pauses ~6 s between Gemini calls to stay under free-tier rate limits. The pause is **not** counted in
  latency. Set `False` with a paid key. (Free-tier prompts may be used by Google to improve its products; our tickets are synthetic.)
- `HYBRID_THRESHOLD`: escalate to Gemini when the decision model's **least confident field** is below this.
  At 0.7, XGBoost escalates about 20% of tickets: the ones it gets wrong most often.
- `USE_LAYA`: also test fine-tuned Laya (needs a GPU runtime and the checkpoint on Drive from notebook 02).
- `CPU_USD_PER_HOUR` / `GPU_USD_PER_HOUR`: rough cloud prices to turn local inference time into dollars.

In [ ]:
GEMINI_MODEL = "gemini-3.1-flash-lite"
N_TICKETS = 100
FREE_TIER = True
HYBRID_THRESHOLD = 0.7
USE_LAYA = False
LAYA_DIR = "/content/drive/MyDrive/typed-decision-models/laya-finetuned"

# Assumed on-demand prices for a small CPU VM and a T4 GPU VM. Edit to match your cloud.
CPU_USD_PER_HOUR = 0.04
GPU_USD_PER_HOUR = 0.35

PAUSE_S = 6.0 if FREE_TIER else 0.0

## 3. Data and the decision model

Same splits as every other notebook. The ticket sample is fixed (seed 42), so every router sees exactly the same tickets.

In [ ]:
from src.dataset import load_splits
from src.models.classical_ml import ClassicalDecisionModel

train_df, val_df, test_df = load_splits("data")
sample = test_df.sample(n=min(N_TICKETS, len(test_df)), random_state=42).reset_index(drop=True)

xgb = ClassicalDecisionModel("xgb").fit(train_df)

laya = None
if USE_LAYA:
    %pip install -q laya
    os.environ["USE_TF"] = "0"
    from google.colab import drive
    drive.mount("/content/drive")
    import torch
    from src.models.laya_model import LayaDecisionModel
    laya = LayaDecisionModel(LAYA_DIR, name="laya:finetuned",
                             device="cuda" if torch.cuda.is_available() else "cpu").load()
print(f"{len(sample)} tickets | decision models:", [m.name for m in [xgb, laya] if m is not None])

## 4. The Gemini router

`GeminiDecisionModel` (in `src/models/llm_router.py`) sends a **system prompt** (team descriptions, the routing policy in
plain words, 12 labelled examples, two per team) plus the ticket as JSON. Gemini must reply with JSON that matches a
schema (enums for the team and priority), so there's no free-text parsing. Thinking is set to `minimal`: a router doesn't
need long reasoning, and thinking tokens are billed as output.

A fair setup: the LLM gets the same information a human would get on day one (the policy and some examples), while the
trained models learned from the 2,099 training tickets. Here's the start of the prompt:

In [ ]:
from src.models.llm_router import GeminiDecisionModel, pick_examples

examples = pick_examples(train_df, per_route=2)


def new_gemini():
    """A fresh Gemini router, so each experiment tracks its own calls and cost."""
    return GeminiDecisionModel(GEMINI_MODEL, examples=examples, min_seconds_between_calls=PAUSE_S)


gemini = new_gemini()
print(gemini.system_prompt[:1500], "...")
print(f"\nsystem prompt: {len(gemini.system_prompt):,} characters")

One test call, to see the decision and what it cost.

In [ ]:
probe = new_gemini()
answer = probe.predict_one(sample.iloc[0])
print("ticket :", sample.iloc[0]["query"])
print("gemini :", answer.route, answer.priority, "human =", answer.requires_human, f"({answer.latency_ms:.0f} ms)")
print("truth  :", sample.iloc[0]["route_target"], sample.iloc[0]["priority_level"], "human =", sample.iloc[0]["requires_human"])
probe.calls[0]

## 5. Hybrid routers

`HybridDecisionModel` (in `src/models/hybrid.py`) runs the decision model first and calls Gemini only when the decision
model's weakest field confidence is below `HYBRID_THRESHOLD`. Each hybrid gets its own Gemini instance so its cost is
counted separately.

In [ ]:
from src.models.hybrid import HybridDecisionModel

routers = {
    "XGBoost": xgb,
    "Gemini only": gemini,
    "Hybrid XGBoost→Gemini": HybridDecisionModel(xgb, new_gemini(), threshold=HYBRID_THRESHOLD),
}
if laya is not None:
    routers["Laya (fine-tuned)"] = laya
    routers["Hybrid Laya→Gemini"] = HybridDecisionModel(laya, new_gemini(), threshold=HYBRID_THRESHOLD)
list(routers)

## 6. Run every router through the same graph

`build_router_graph` (in `src/router_graph.py`) is the graph from notebook 04 with the router model plugged in; `run_tickets`
sends each ticket through it once and records where it landed and how long the router and the whole graph took.
With `FREE_TIER = True` the Gemini runs take a while because of the pauses between calls.

In [ ]:
import time
from src.router_graph import build_router_graph, run_tickets

runs = {}
for label, model in routers.items():
    start = time.time()
    graph = build_router_graph(model, confidence_floor=0.5)
    runs[label] = run_tickets(graph, sample)
    print(f"{label:<24} done in {time.time() - start:6.1f} s wall-clock")

## 7. Results table

- **Cost per 1k tickets** = Gemini tokens × price, plus local compute time × the hourly price above (that part is an upper
  bound: it assumes the machine does nothing else). Hosted LLM cost has no machine to pay for, the decision models do.
- **LLM share** = fraction of tickets that called Gemini.
- **Failed answers** = Gemini replies that didn't match the schema (those tickets go to a human).

In [ ]:
import numpy as np
import pandas as pd

ON_GPU = {"Laya (fine-tuned)", "Hybrid Laya→Gemini"}
# Which decision model runs locally inside each router (a hybrid runs its decision model on every ticket).
LOCAL_PART = {"XGBoost": "XGBoost", "Hybrid XGBoost→Gemini": "XGBoost",
              "Laya (fine-tuned)": "Laya (fine-tuned)", "Hybrid Laya→Gemini": "Laya (fine-tuned)"}


def llm_part(model):
    """The Gemini instance inside a router (or None) and the share of tickets it handled."""
    if isinstance(model, GeminiDecisionModel):
        return model, 1.0
    if isinstance(model, HybridDecisionModel):
        return model.slow, model.escalation_rate()
    return None, 0.0


rows = []
for label, res in runs.items():
    model = routers[label]
    llm, llm_share = llm_part(model)
    hourly = GPU_USD_PER_HOUR if label in ON_GPU else CPU_USD_PER_HOUR
    # Local compute: the decision model's own time per ticket, at the machine's hourly price.
    local_ms = runs[LOCAL_PART[label]]["router_ms"].mean() if label in LOCAL_PART else 0.0
    local_cost_1k = local_ms / 1000 / 3600 * hourly * 1000
    usage = llm.usage_summary() if llm is not None else {}
    llm_cost_1k = usage.get("total_cost_usd", 0.0) / len(res) * 1000
    rows.append({
        "router": label,
        "correct_destination": res["correct_destination"].mean(),
        "exact_match": res["exact"].mean(),
        "router_p50_ms": np.percentile(res["router_ms"], 50),
        "router_p95_ms": np.percentile(res["router_ms"], 95),
        "graph_p50_ms": np.percentile(res["graph_ms"], 50),
        "graph_p95_ms": np.percentile(res["graph_ms"], 95),
        "llm_share": llm_share,
        "avg_input_tokens": usage.get("avg_input_tokens", 0.0),
        "avg_output_tokens": usage.get("avg_output_tokens", 0.0),
        "failed_answers": usage.get("failed_parses", 0),
        "usd_per_1k_tickets": llm_cost_1k + local_cost_1k,
    })

summary = pd.DataFrame(rows).set_index("router")
os.makedirs("artifacts", exist_ok=True)
summary.to_csv("artifacts/results_llm_vs_decision.csv")
summary.round(4)

## 8. Charts

Same colour per router in every chart. Latency and cost use **log scales**, because the routers differ by orders of
magnitude.

In [ ]:
import matplotlib.pyplot as plt

ROUTER_COLORS = {
    "XGBoost": "#eb6834",
    "Laya (fine-tuned)": "#008300",
    "Gemini only": "#6250d6",
    "Hybrid XGBoost→Gemini": "#1baf7a",
    "Hybrid Laya→Gemini": "#2a78d6",
}
TEXT_COLOR, GRID_COLOR = "#3d3d3a", "#e4e3dd"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRID_COLOR, "axes.labelcolor": TEXT_COLOR,
                     "xtick.color": TEXT_COLOR, "ytick.color": TEXT_COLOR, "axes.titlecolor": TEXT_COLOR})
order = list(summary.index)


def tidy(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)


fig, (left, right) = plt.subplots(1, 2, figsize=(15, 0.6 * len(order) + 1.6), sharey=True)

# Left: router latency, dot = p50, line end = p95
for y, label in enumerate(order):
    p50, p95 = summary.loc[label, "router_p50_ms"], summary.loc[label, "router_p95_ms"]
    color = ROUTER_COLORS.get(label, "#898781")
    left.plot([p50, p95], [y, y], color=color, linewidth=2, solid_capstyle="round")
    left.scatter([p50], [y], s=70, color=color, edgecolor="white", linewidth=2, zorder=3)
    left.text(p95 * 1.15, y, f"{p50:,.0f} / {p95:,.0f} ms", va="center", fontsize=9, color=TEXT_COLOR)
left.set_yticks(range(len(order)), order)
left.set_xscale("log")
left.set_xlim(summary["router_p50_ms"].min() / 2, summary["router_p95_ms"].max() * 15)
left.set_title("Router latency, p50 / p95 (log scale)", loc="left", fontsize=11)
tidy(left)

# Right: dollars per 1,000 tickets
costs = summary["usd_per_1k_tickets"].values
right.barh(order, costs, color=[ROUTER_COLORS.get(l, "#898781") for l in order], height=0.6)
for y, v in enumerate(costs):
    right.text(v * 1.15, y, f"${v:.4f}", va="center", fontsize=9, color=TEXT_COLOR)
right.set_xscale("log")
right.set_xlim(costs.min() / 2, costs.max() * 15)
right.set_title("Cost per 1,000 tickets, USD (log scale)", loc="left", fontsize=11)
tidy(right)

left.invert_yaxis()
fig.tight_layout()
plt.show()

Quality vs cost: upper-left is best (more right answers for less money).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label in order:
    x, y = summary.loc[label, "usd_per_1k_tickets"], summary.loc[label, "exact_match"]
    ax.scatter([x], [y], s=90, color=ROUTER_COLORS.get(label, "#898781"), edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(label, (x, y), xytext=(8, 4), textcoords="offset points", fontsize=9, color=TEXT_COLOR)
ax.set_xscale("log")
ax.set_xlabel("USD per 1,000 tickets (log scale)")
ax.set_ylabel("exact match")
ax.set_ylim(0, 1.05)
ax.set_title("Exact match vs cost", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 9. What does it look like at scale?

Projected monthly cost and the time spent waiting on the router, at 1 million tickets a month. Latency adds up as
user-facing wait time even when money doesn't.

In [ ]:
TICKETS_PER_MONTH = 1_000_000

projection = pd.DataFrame({
    "usd_per_month": summary["usd_per_1k_tickets"] * TICKETS_PER_MONTH / 1000,
    "router_hours_per_month": summary["router_p50_ms"] * TICKETS_PER_MONTH / 1000 / 3600,
    "wrong_destinations_per_month": (1 - summary["correct_destination"]) * TICKETS_PER_MONTH,
})
projection.round(0).astype(int)

## 10. Where do Gemini and XGBoost disagree?

Tickets where exactly one of them picked the right destination. This shows what kind of mistakes each one makes:
the LLM reads language well but has to guess the numeric policy; the trees learned the thresholds from data.

In [ ]:
both = sample[["query", "account_tier", "urgency_score", "past_failures", "route_target", "priority_level", "requires_human"]].copy()
both["xgb_dest"] = runs["XGBoost"]["handled_by"]
both["gemini_dest"] = runs["Gemini only"]["handled_by"]
both["should_be"] = runs["XGBoost"]["expected"]
xgb_only = both[(both["xgb_dest"] == both["should_be"]) & (both["gemini_dest"] != both["should_be"])]
gemini_only = both[(both["gemini_dest"] == both["should_be"]) & (both["xgb_dest"] != both["should_be"])]
print(f"XGBoost right, Gemini wrong: {len(xgb_only)} | Gemini right, XGBoost wrong: {len(gemini_only)}")
pd.concat([xgb_only.head(5), gemini_only.head(5)])

Priority is where they differ most. A confusion table for Gemini's priority calls:

In [ ]:
priority_order = ["low", "medium", "high", "critical"]
pd.crosstab(sample["priority_level"], runs["Gemini only"]["priority"],
            rownames=["true"], colnames=["gemini"]).reindex(index=priority_order, columns=priority_order, fill_value=0)

**Takeaways to fill in after running:**
- Exact match: XGBoost _TODO_ vs Gemini only _TODO_ vs hybrid _TODO_
- Router p50 latency: XGBoost _TODO_ ms vs Gemini _TODO_ ms (_TODO_x)
- Cost per 1M tickets: XGBoost $_TODO_ vs Gemini $_TODO_ vs hybrid $_TODO_
- Did the hybrid beat both on quality? What share of tickets did it send to Gemini? _TODO_

Caveat: Gemini latency includes the network round trip from Colab, and API latency varies by time of day. Run it twice
if the numbers matter.